# Dacon 이커머스 고객 분석 — 프로젝트 요약

## 분석 배경

CRM/마케팅팀으로부터 "고객 재구매율을 높이기 위해 어떤 고객군을 우선 관리해야 하는가"라는 분석 요청을 받았다는 상황을 가정했다.

이 프로젝트는 2019년 온라인 커머스 거래 데이터를 기반으로 ETL, EDA, 리텐션, RFM 세그먼테이션, 등급별 심층 분석을 연결해 고객 관리 우선순위를 도출한다.

## 전체 분석 흐름

| 순서 | 노트북 | 역할 |
|------|--------|------|
| 00 | ETL 파이프라인 | 원본 5개 테이블 정제, 조인, `orders_master` 생성 |
| 01 | EDA | 매출, 카테고리, 고객, 쿠폰, 마케팅, 지역, 요일, 장바구니 패턴 탐색 |
| 02 | 리텐션 분석 | 코호트 리텐션, 마케팅 비용, 카테고리별 재구매 주기 분석 |
| 03 | RFM 세그먼테이션 | R/F/M 점수화, PCA 가중치, 등급 및 행동 세그먼트 배정 |
| 04 | Bronze 분석 | 재활성화 가능성과 Silver 전환 가능성 분석 |
| 05 | Silver 분석 | 이탈 구조와 Gold 전환 가능성 분석 |
| 06 | Gold 분석 | 이탈 위험과 Platinum 전환 경로 분석 |
| 07 | Diamond/Platinum 분석 | 상위 고객 재구매 패턴과 유지 전략 분석 |

전체 흐름은 `거래 데이터 정제 -> 고객 행동 지표화 -> 리텐션 원인 파악 -> RFM 등급화 -> 등급별 액션 도출` 순서로 구성했다.

## 파일별 핵심 인사이트

| 파일 | 핵심 인사이트 | 다음 분석으로 이어지는 의미 |
|------|---------------|---------------------------|
| 00_ETL | 거래, 고객, 할인, 마케팅, 세금 데이터를 통합해 고객 단위 분석이 가능한 `orders_master`를 구성했다. | 이후 모든 분석에서 동일한 기준의 거래·고객 데이터를 사용한다. |
| 01_EDA | Nest-USA 매출 집중, 여성 고객 매출 기여, Chicago/California 지역 집중, 주중 중후반 구매 집중이 확인됐다. | 고객 단위 리텐션과 RFM 분석으로 패턴을 심화할 필요가 있다. |
| 02_리텐션 | 전체 코호트 리텐션이 낮고, 첫 구매 직후 이탈 방어가 주요 과제로 나타났다. | 단순 매출보다 고객 재방문과 재참여 우선순위를 봐야 한다. |
| 03_RFM | R/F/M 점수와 PCA 가중치로 고객을 5개 등급과 11개 행동 세그먼트로 나눴다. | 등급별로 서로 다른 질문을 설정할 수 있는 분석 기준이 마련됐다. |
| 04_Bronze | 고객 수는 많지만 재방문율과 1인당 지출이 낮고, 휴면·신규 고객 관리가 핵심이다. | Bronze는 첫 구매 유지와 재활성화 캠페인이 우선이다. |
| 05_Silver | 안정 성장군과 이탈군이 공존하며, 잠재 충성 고객은 Gold 전환에 가장 가깝다. | Silver는 Gold 전환과 이탈 위험 고객 재참여를 동시에 관리해야 한다. |
| 06_Gold | 매출 기여가 큰 핵심 중간층이지만 장기 공백과 상위 등급 대비 재방문율 격차가 존재한다. | Gold는 이탈 방어와 Platinum 전환 경로를 세그먼트별로 분리해야 한다. |
| 07_D/P | Diamond/Platinum은 매출 기여가 높지만, Platinum 내부에 고가치 이탈 조짐 고객이 존재한다. | 상위 고객은 할인보다 전용 혜택, 등급 전환 알림, 개인화 재참여가 중요하다. |

## RFM 등급 설계 요약

RFM 점수는 Recency, Frequency, Monetary를 각각 1-5점으로 변환한 뒤 PCA 기반 가중치를 적용해 20-100점 범위로 산출했다.

| 지표 | 점수화 방식 | 해석 |
|------|-------------|------|
| Recency | 분위수 기반 5등분, 최근일수록 고점 | 최근 구매 여부 |
| Frequency | IQR 기반 5등분 | 구매 빈도 |
| Monetary | IQR 기반 5등분 | 누적 구매금액 |

| 등급 | 기준 | 분석 목적 |
|------|------|-----------|
| Diamond | ≥ 95 | 최상위 VIP 유지 |
| Platinum | ≥ 80 | 고가치 고객 유지 및 Diamond 전환 |
| Gold | ≥ 65 | 이탈 방어와 Platinum 전환 |
| Silver | ≥ 50 | Gold 전환과 안정군 성장 |
| Bronze | < 50 | 신규 온보딩과 재활성화 |

고정 컷오프를 사용한 이유는 등급 기준을 비즈니스 커뮤니케이션에서 쉽게 설명하고, 등급별 후속 분석 질문을 명확히 나누기 위해서다.

## 등급별 인사이트 요약

| 등급 | 핵심 상태 | 주요 발견 | 우선 액션 |
|------|-----------|-----------|-----------|
| Bronze | 고객 수는 많지만 재방문율과 1인당 지출이 낮음 | 신규·휴면 고객 비중이 높고, Silver 전환은 F/M 개선이 핵심 | 첫 구매 후 2차 구매 유도, 휴면 고객 재활성화 |
| Silver | 성장 가능군과 이탈군이 공존 | 잠재 충성 고객은 Gold에 가깝고, 이탈 위험 고객도 고가치 | Gold 전환 부스터, 이탈 위험 개인화 재참여 |
| Gold | 매출 기여가 큰 핵심 중간층 | 상위 등급 대비 재방문율 격차와 장기 공백 경험이 존재 | 이탈 방어, Platinum 전환 경로 분리 |
| Platinum | 고가치 안정 고객 | 일부 이탈 조짐 고객이 일반 Platinum보다 구매금액이 높음 | Recency 회복, Diamond 전환 알림 |
| Diamond | 최상위 VIP 고객 | 재방문율과 1인당 지출이 가장 높은 고객군 | 전용 혜택, VIP 경험 강화 |

## 전체 핵심 발견

1. **매출은 소수 카테고리와 상위 고객군에 집중되어 있다.**  
   Nest-USA가 전체 매출의 약 54%를 차지하고, Diamond/Platinum은 전체 고객의 약 14%지만 매출의 약 49%를 담당한다.

2. **첫 구매 이후 이탈 방어가 핵심 과제다.**  
   전체 코호트 리텐션은 낮고, +1개월 복귀율이 전 구간에서 낮게 나타나 신규 고객 온보딩이 중요하다.

3. **고객 가치는 Recency만으로 설명되지 않는다.**  
   Silver, Gold, Platinum에서 최근 구매가 줄어든 고객 중에도 구매금액이 높은 고객군이 확인된다.

4. **등급별 관리 질문이 다르다.**  
   Bronze는 재활성화, Silver는 Gold 전환, Gold는 이탈 방어와 Platinum 전환, Diamond/Platinum은 유지 전략이 핵심이다.

5. **단일 캠페인보다 세그먼트별 타이밍 분리가 필요하다.**  
   이탈 시점과 전환 가능성이 등급·세그먼트마다 달라, 동일 메시지를 전체 고객에게 보내는 방식은 효율이 낮을 수 있다.

## 추천 액션

| 우선순위 | 대상 | 액션 방향 |
|----------|------|-----------|
| 1 | 신규·Bronze 고객 | 첫 구매 후 14-30일 내 2차 구매 리마인더 |
| 2 | Bronze 휴면 고객 | 마지막 구매 후 90-150일 구간 재활성화 캠페인 |
| 3 | Silver 잠재 충성 고객 | Gold 전환 진행률 알림과 F/M 부스터 |
| 4 | Gold 이탈 위험군 | 7-8월 이탈 집중 전 선제 접촉 |
| 5 | Platinum 이탈 조짐 고객 | 과거 구매 품목 기반 개인화 재참여 |
| 6 | Diamond 고객 | 할인보다 전용 혜택과 등급 유지 경험 강화 |

액션은 전체 고객 대상 일괄 캠페인보다, 등급과 세그먼트별 상태에 맞춰 메시지·타이밍·혜택을 분리하는 방향이 적합하다.

## 한계와 다음 단계

- 데이터가 2019년 1개년 기준이므로 계절성 반복 여부는 다년도 데이터로 재검증이 필요하다.
- 캠페인 효율은 실제 발송 비용, 마진, 전환율이 포함된 실험 데이터가 있어야 정밀하게 평가할 수 있다.
- RFM 등급은 고객 행동을 설명하는 운영 기준이며, 최종 캠페인 우선순위는 재방문 가능성, 예상 구매금액, 비용을 함께 고려해야 한다.
- 다음 단계에서는 Tableau 대시보드로 등급별 고객 수, 매출 기여, 재방문율, 세그먼트별 액션 우선순위를 한 화면에서 확인할 수 있게 구성한다.